# Session 7: time, many charts, and the one that matters

Notebook 01 drew one chart at a time. Real questions are messier: they run
over time, they come in eight groups at once, and the reader needs to be
told where to look. This notebook covers:

1. real dates on the x-axis
2. reference lines, shaded spans and annotations
3. many lines: the spaghetti problem, and small multiples
4. highlighting: grey for everything, colour for the one that matters
5. colour that works for every reader
6. one figure with four panels, saved for someone else

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
os.makedirs("output", exist_ok=True)

plays = pd.read_csv("data/clean/plays.csv")
plays["date"] = pd.to_datetime(plays["played_at"])

BLUE = "#4f6ddb"          # the course colour: "this is the data"
GREY = "#b8bec9"          # context: there, but not the point
HIGHLIGHT = "#d55e00"     # the one thing the title is about

print(plays["date"].min().date(), "to", plays["date"].max().date())

Three colours, named once at the top, used everywhere below. A chart where
blue means the same thing in every panel is easier to read than one where
matplotlib picked the colours.

---

# 1. Real dates

In notebook 01 the months were text, `"2024-01"`. That works, but
matplotlib sees 24 labels in a row and has no idea they are time. Real
dates fix that. `resample("MS")` is the session 6 tool: one row per month
(MS means "month start"), counting the plays in each.

In [ ]:
monthly = plays.set_index("date").resample("MS").size()
monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, marker="o", color=BLUE)
ax.set_ylabel("plays per month")
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

No labels to turn sideways: matplotlib knows these are dates, so it picks a
sensible number of ticks and spaces them by time. If a month were missing,
the gap would show as a longer stretch of line instead of being squeezed
out, which text labels would do without telling you.

# 2. Telling the reader where to look

The line has a shape. Three tools point at it:

* **a reference line**, `ax.axhline(y)`: a horizontal line across the
  chart, here at the average month, so every point reads as above or below
* **a shaded span**, `ax.axvspan(start, end)`: a band, here over each
  summer
* **an annotation**, `ax.annotate(text, xy=point, xytext=offset)`: a label
  with an arrow to one point

In [ ]:
average = monthly.mean()
peak_date, peak = monthly.idxmax(), monthly.max()
low_date, low = monthly.idxmin(), monthly.min()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly.index, monthly.values, marker="o", color=BLUE)

ax.axhline(average, color=GREY, linestyle="--", linewidth=1)
ax.text(monthly.index[0], average + 3, f"average: {average:.1f} plays a month", color="#6b7280")

for year in [2024, 2025]:
    ax.axvspan(pd.Timestamp(f"{year}-06-01"), pd.Timestamp(f"{year}-08-01"),
               color=GREY, alpha=0.25, linewidth=0)

ax.annotate(f"{peak_date:%B %Y}: {peak}", xy=(peak_date, peak), xytext=(15, 8),
            textcoords="offset points", arrowprops={"arrowstyle": "->"})
ax.annotate(f"{low_date:%B %Y}: {low}", xy=(low_date, low), xytext=(15, -22),
            textcoords="offset points", arrowprops={"arrowstyle": "->"})

ax.set_title(f"Every summer falls below the average of {average:.1f} plays a month")
ax.set_ylabel("plays per month")
ax.set_ylim(0, 145)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

`f"{peak_date:%B %Y}"` is one more f-string format: after the colon, a date
takes date codes. `%B` is the month name and `%Y` the year, so the label
says "October 2024" instead of "2024-10-01 00:00:00".

`xy` is the point the arrow touches, in data units: a date and a number of
plays. `xytext=(15, 8)` with `textcoords="offset points"` puts the text a
little to the right and above it, measured in points, so it never lands on
the line.

Check the title before trusting it:

In [ ]:
summers = monthly[monthly.index.month.isin([6, 7, 8])]
print(summers)

assert (summers < average).all()
assert round(average, 1) == 91.0
assert (f"{peak_date:%B %Y}", peak) == ("October 2024", 127)
assert (f"{low_date:%B %Y}", low) == ("August 2025", 57)

Six summer months, all below 91.0. The title is true.

## The same months, one year on top of the other

"Is this year like last year?" is a question about the **month of the
year**, not the date. Put the month number along the bottom and draw one
line per year. `pivot_table` makes the table, the session 6 way.

In [ ]:
by_year = plays.pivot_table(index=plays["date"].dt.month, columns=plays["date"].dt.year,
                            values="play_id", aggfunc="count")
by_year

In [ ]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(by_year.index, by_year[2024], color=GREY, linewidth=2)
ax.plot(by_year.index, by_year[2025], color=BLUE, linewidth=3)

ax.text(12.15, by_year.loc[12, 2024], "2024", color="#6b7280", va="center")
ax.text(12.15, by_year.loc[12, 2025], "2025", color=BLUE, va="center", fontweight="bold")

ax.set_xticks(range(1, 13), month_names)
ax.set_title("2025 had a quieter summer than 2024, and a busier autumn")
ax.set_ylabel("plays per month")
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Two things are new.

* **No legend.** The year is written at the end of each line, where the eye
  already is. A legend makes the reader look away, match a colour, and look
  back. With two or three lines, label them directly
* **2024 is grey.** It is there for comparison, so it steps back. The year
  the title talks about gets the colour and the thicker line

`ax.set_xticks(positions, labels)` puts month names where the numbers 1 to
12 were. Check the title:

In [ ]:
summer = by_year.loc[[6, 7, 8]].sum()
autumn = by_year.loc[[10, 11, 12]].sum()
print("June to August:   ", summer.to_dict())
print("October to December:", autumn.to_dict())

assert summer[2025] < summer[2024]
assert autumn[2025] > autumn[2024]
assert (summer[2024], summer[2025], autumn[2024], autumn[2025]) == (201, 175, 313, 337)

---

# 3. Many lines

**How did each genre do over the two years?** Monthly counts per genre are
small and jumpy, so use quarters. `.dt.to_period("Q")` turns each date into
its quarter, and `.dt.start_time` turns that back into a real date: the
first day of the quarter.

In [ ]:
plays["quarter"] = plays["date"].dt.to_period("Q").dt.start_time
quarterly = plays.pivot_table(index="quarter", columns="genre",
                              values="play_id", aggfunc="count")
quarterly

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
quarterly.plot(ax=ax)
ax.set_ylabel("plays per quarter")
plt.show()

Eight lines, eight colours, one legend, and a tangle in the bottom half.
Which one is Folk? You have to look at the legend, find the colour, and
follow a line through four crossings. This is called a **spaghetti chart**,
and pandas will draw you one in a single line.

Two ways out: draw each line on its own little chart, or keep them together
and make all but one of them step back.

## Small multiples

`plt.subplots(2, 4)` makes two rows of four charts. `axes` is then a grid of
eight `ax` objects; `axes.flat` lists them one by one, so `zip` can pair each
chart with a genre.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.5), sharex=True, sharey=True)

for ax, genre in zip(axes.flat, quarterly.columns):
    ax.plot(quarterly.index, quarterly[genre], color=BLUE, linewidth=2)
    ax.set_title(genre)
    ax.set_xticks([pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")], ["2024", "2025"])
    ax.spines[["top", "right"]].set_visible(False)

for ax in axes[:, 0]:
    ax.set_ylabel("plays per quarter")
axes[0, 0].set_ylim(bottom=0)

fig.suptitle("Electronic and Folk are the two biggest genres in every quarter")
plt.show()

Every genre gets its own chart, in the same place, with the same axes. Now
the eye does two things it is good at: read one shape at a time, and
compare heights across the grid.

**`sharey=True` is what makes the comparison honest.** All eight charts use
the same y-axis, so a tall line really is more plays. Try it: change it to
`sharey=False` and run the cell again. Every chart stretches to fill its
own box, Rock's line, which never goes above 24, looks as big as
Electronic's, which never goes below 47, and the grid is lying to you.
Small multiples that are meant to be compared must share an axis.

`sharex=True` does the same for the dates, and also hides the repeated
year labels on the top row. `axes[:, 0]` is the first column of the grid,
the only charts that need a y-label. Because the y-axis is shared, setting
its bottom to zero once, after everything is drawn, sets it for all eight.

Check the title:

In [ ]:
top_two = quarterly.apply(lambda row: set(row.nlargest(2).index), axis=1)
print(top_two)

assert (top_two == {"Electronic", "Folk"}).all()
assert quarterly["Rock"].max() == 24
assert quarterly["Electronic"].min() == 47

`quarterly.apply(..., axis=1)` runs a small function on each row, here
picking the two largest. `lambda row: ...` is a function with no name,
written in one line, for exactly this kind of job. You can always write it
with `def` instead.

---

# 4. Highlighting: grey everything, colour the one that matters

Small multiples show every genre. Often the question is about one of them.
**Which genre lost the most plays from 2024 to 2025?**

In [ ]:
per_year = plays.pivot_table(index="genre", columns=plays["date"].dt.year,
                             values="play_id", aggfunc="count")
per_year["change"] = per_year[2025] - per_year[2024]
per_year.sort_values("change")

Folk, down 34. Now draw it so the reader sees that in a second, with every
other genre still there for scale:

In [ ]:
loser = per_year["change"].idxmin()
before, after = per_year.loc[loser, 2024], per_year.loc[loser, 2025]
last = quarterly.index[-1]

fig, ax = plt.subplots(figsize=(10, 4.5))

for genre in quarterly.columns:
    if genre != loser:
        ax.plot(quarterly.index, quarterly[genre], color=GREY, linewidth=1.5)

ax.plot(quarterly.index, quarterly[loser], color=HIGHLIGHT, linewidth=3)
ax.text(last + pd.Timedelta(days=20), quarterly.loc[last, loser], loser,
        color=HIGHLIGHT, va="center", fontweight="bold")
ax.text(last + pd.Timedelta(days=20), quarterly.loc[last, "Electronic"], "Electronic",
        color="#6b7280", va="center")

spring_2024 = quarterly.loc[pd.Timestamp("2024-04-01"), loser]
spring_2025 = quarterly.loc[pd.Timestamp("2025-04-01"), loser]
ax.annotate(f"spring 2025: {spring_2025} plays\nspring 2024: {spring_2024}",
            xy=(pd.Timestamp("2025-04-01"), spring_2025),
            xytext=(pd.Timestamp("2025-04-01"), 98), ha="center", color=HIGHLIGHT,
            arrowprops={"arrowstyle": "->", "color": HIGHLIGHT})

ax.set_title(f"{loser} lost the most plays: {before} in 2024, {after} in 2025")
ax.set_ylabel("plays per quarter")
ax.set_ylim(0, 115)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
assert loser == "Folk"
assert (before, after) == (236, 202)
assert (spring_2024, spring_2025) == (58, 39)

The same eight lines as the spaghetti chart. The only differences are the
colours and two labels, and now there is nothing to decode: the title names
Folk, and Folk is the only line with colour.

The recipe, for any chart with too many things in it:

1. draw everything in grey, thin
2. draw the one the title is about last, so it sits on top, in colour, thick
3. label it directly, and maybe one other line for scale
4. annotate the moment that makes the point, with the text in empty space
   and an arrow to the point. `xytext` can be a date and a number too, like
   `xy`, when you want the text at an exact spot on the chart

---

# 5. Colour that works for every reader

About one man in twelve does not see red and green as clearly different.
Charts are also printed in black and white, shown on bad projectors, and
read on phones in the sun. A few rules cover nearly all of it:

* **colour should mean something.** One colour for the data, grey for
  context, one strong colour for the point. Not a different colour per bar
  "to make it look nice"
* **never colour alone.** Label lines directly, write the numbers on the
  bars. Then colour helps, and nobody depends on it
* **avoid red against green** as the only difference. Blue against orange is
  safe for nearly everyone
* **the colours should differ in lightness**, not only in hue, so they
  survive a greyscale printer

There is a well-known palette built for this, called **Okabe-Ito**. The
first four are worth remembering:

In [ ]:
OKABE_ITO = {
    "orange": "#e69f00",
    "sky blue": "#56b4e9",
    "green": "#009e73",
    "blue": "#0072b2",
    "vermillion": "#d55e00",
    "purple": "#cc79a7",
    "yellow": "#f0e442",
}

fig, ax = plt.subplots(figsize=(9, 1.6))
ax.bar(list(OKABE_ITO), [1] * len(OKABE_ITO), color=list(OKABE_ITO.values()))
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
plt.show()

`HIGHLIGHT` at the top of this notebook is the vermillion.

## The greyscale test

You cannot see your chart the way everybody else does. You can check one
thing, though: whether it still works without colour. The stacked
histogram in notebook 01 used blue and red. Save it, read the PNG back in as
numbers, and turn it grey.

In [ ]:
kept = plays.loc[plays["skipped"] == 0, "minutes_played"]
skips = plays.loc[plays["skipped"] == 1, "minutes_played"]


def save_stacked_histogram(colours, path):
    """Draw the skipped-versus-kept histogram in two colours and save it."""
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist([kept, skips], bins=range(0, 13), stacked=True, color=colours,
            edgecolor="white", label=["played through", "skipped"])
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)


def lightness(hex_colour):
    """Return how light a colour looks, from 0 (black) to 1 (white)."""
    red, green, blue = (int(hex_colour[i:i + 2], 16) / 255 for i in (1, 3, 5))
    return 0.299 * red + 0.587 * green + 0.114 * blue


pairs = {
    "notebook 01": ["#4f6ddb", "#c0392b"],
    "Okabe-Ito": [OKABE_ITO["blue"], OKABE_ITO["orange"]],
}

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for row, (name, colours) in zip(axes, pairs.items()):
    path = f"output/histogram_{name.replace(' ', '_')}.png"
    save_stacked_histogram(colours, path)
    image = plt.imread(path)[:, :, :3]
    grey = image @ [0.299, 0.587, 0.114]

    gap = abs(lightness(colours[0]) - lightness(colours[1]))
    row[0].imshow(image)
    row[0].set_title(f"{name}: in colour")
    row[1].imshow(grey, cmap="gray", vmin=0, vmax=1)
    row[1].set_title(f"{name}: in greyscale, lightness gap {gap:.2f}")
    for ax in row:
        ax.axis("off")

plt.show()

`plt.imread` reads a PNG as a grid of pixels, each one three numbers for
red, green and blue. Multiplying by `[0.299, 0.587, 0.114]` with `@` mixes
them into one lightness per pixel; the weights are there because the eye
sees green as much brighter than blue.

In greyscale the notebook 01 pair is two nearly identical greys: only the
white edges keep the bars apart. The Okabe-Ito pair is a dark grey and a
light one. Same chart, same data, and one version survives the photocopier.

In [ ]:
old_gap = abs(lightness("#4f6ddb") - lightness("#c0392b"))
new_gap = abs(lightness(OKABE_ITO["blue"]) - lightness(OKABE_ITO["orange"]))
print(f"lightness gap: {old_gap:.2f} before, {new_gap:.2f} after")

assert round(old_gap, 2) == 0.07
assert round(new_gap, 2) == 0.29

**Colour for amounts.** When colour itself shows a number, as in a heatmap,
use one colour going from light to dark (matplotlib's `"Blues"` or
`"viridis"`), never a rainbow. A rainbow has no order: nobody knows whether
yellow is more than green.

---

# 6. One figure, four panels, ready to share

Everything comes together here. One figure, four questions, one of each
kind from notebook 01, and a title over all of it that says what the whole
picture adds up to.

Two new arguments:

* `layout="constrained"` makes matplotlib move the panels so no title or
  label overlaps another
* `dpi=200` on `savefig`, for a document that might be printed. 100 is fine
  for a screen, 150 for a slide

In [ ]:
by_genre = plays.groupby("genre")["minutes_played"].mean().sort_values()
per_artist = plays.groupby("artist_name").agg(plays=("play_id", "count"),
                                              minutes=("minutes_played", "sum"))
blue, orange = OKABE_ITO["blue"], OKABE_ITO["orange"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained")
(time_ax, genre_ax), (spread_ax, artist_ax) = axes

time_ax.plot(monthly.index, monthly.values, marker="o", color=blue)
time_ax.axhline(average, color=GREY, linestyle="--", linewidth=1)
time_ax.set_title(f"Summers fall below the {average:.1f}-play average")
time_ax.set_ylabel("plays per month")
time_ax.set_ylim(bottom=0)

colours = [orange if genre == "Jazz" else blue for genre in by_genre.index]
bars = genre_ax.barh(by_genre.index, by_genre.values, color=colours)
genre_ax.bar_label(bars, fmt="{:.1f}", padding=3)
genre_ax.set_title(f"Jazz plays last longest: {by_genre['Jazz']:.1f} minutes")
genre_ax.set_xlabel("average minutes per play")

spread_ax.hist([kept, skips], bins=range(0, 13), stacked=True, color=[blue, orange],
               edgecolor="white", label=["played through", "skipped"])
spread_ax.set_title(f"{(skips < 1).mean():.1%} of skips last under a minute")
spread_ax.set_xlabel("minutes played")
spread_ax.set_ylabel("number of plays")
spread_ax.legend()

artist_ax.scatter(per_artist["plays"], per_artist["minutes"], color=blue, alpha=0.75)
artist_ax.annotate("Glass Tram", (per_artist.loc["Glass Tram", "plays"],
                                  per_artist.loc["Glass Tram", "minutes"]),
                   xytext=(-10, 0), textcoords="offset points", ha="right")
artist_ax.set_title("More plays, more minutes, but not in step")
artist_ax.set_xlabel("plays per artist")
artist_ax.set_ylabel("total minutes")
artist_ax.set_ylim(bottom=0)

for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Two years of listening: {len(plays):,} plays, "
             f"{plays['minutes_played'].sum():,.0f} minutes", fontsize=16)

fig.savefig("output/listening_overview.png", dpi=200, bbox_inches="tight")
plt.show()

Read it as a reader would: the big title says what this is about, each
panel title gives one answer, and **blue means "the data" in all four
panels**, with orange for the one thing each panel points at. A reader who
has worked out one panel has worked out the colours for all of them.

`(time_ax, genre_ax), (spread_ax, artist_ax) = axes` unpacks the 2-by-2
grid into four names, row by row, so every line below says which panel it
draws on. With four panels, `ax.` style is not a preference any more.

In [ ]:
saved = plt.imread("output/listening_overview.png")
height, width = saved.shape[:2]
print(f"{width} by {height} pixels")

assert len(plays) == 2183
assert round(plays["minutes_played"].sum()) == 8980
assert round((skips < 1).mean() * 100, 1) == 73.6
assert width > 2000

More than 2,000 pixels wide: sharp on a slide, sharp on paper. Open
`output/listening_overview.png` and look at it at full size. If anything
is cramped, make `figsize` bigger rather than the fonts smaller.

**One more format.** `fig.savefig("output/listening_overview.svg")` saves
the same figure as a drawing rather than a grid of pixels. It stays sharp at
any size, which is what you want for print or a web page. PNG is for
documents and slides.

## Next

`03-makeover-exercise.ipynb`: a bad chart, taken apart and rebuilt one
decision at a time.